In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score,  mean_absolute_error

In [3]:
# Load dataset
df = pd.read_csv("comprehensive_risk_indian_roads_dataset.csv")

# Features and target
features = ['hour', 'is_weekend']
cat_features = ['traffic_density', 'weather', 'cause']
target = 'risk_score'

# Clean data
df = df.dropna(subset=features + cat_features + [target])
print(f"Dataset ready: {len(df)} rows")

Dataset ready: 20000 rows


In [5]:
label_encoders = {}
for col in cat_features:
    le = LabelEncoder()
    df[f'{col}_enc'] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"Encoded {col}: {le.classes_}")

Encoded traffic_density: ['high' 'low' 'medium']
Encoded weather: ['clear' 'fog' 'rain']
Encoded cause: ['distraction' 'drunk driving' 'overspeeding' 'poor road' 'weather']


In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

X = df[['hour', 'is_weekend', 'traffic_density_enc', 'weather_enc', 'cause_enc']]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "R²": r2_score(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MAE": mean_absolute_error(y_test, y_pred)
    })

results_df = pd.DataFrame(results)

print(results_df)

               Model        R²      RMSE       MAE
0  Linear Regression  0.293482  0.137325  0.112460
1      Decision Tree  0.772162  0.077983  0.062448
2      Random Forest  0.778331  0.076920  0.061700
3  Gradient Boosting  0.798417  0.073353  0.059295


In [9]:
"""Random Forest was selected for further tuning because it is less sensitive to  
hyperparameters, more robust to overfitting due to its averaging of independent 
trees, and faster to train and iterate on."""

'Random Forest was selected for further tuning because it is less sensitive to  \nhyperparameters, more robust to overfitting due to its averaging of independent \ntrees, and faster to train and iterate on.'

In [11]:
from sklearn.model_selection import GridSearchCV
X = df[['hour', 'is_weekend', 'traffic_density_enc', 'weather_enc', 'cause_enc']]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
params = {
    'n_estimators': [50,100,200],
    'max_depth': [5,10,15]
}

grid = GridSearchCV(
    RandomForestRegressor(),
    param_grid=params,
    cv=5
)

grid.fit(X_train, y_train)

print(grid.best_params_)

{'max_depth': 10, 'n_estimators': 200}


In [13]:
# Random Forest
rf_model = RandomForestRegressor(n_estimators=200, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)


In [15]:
# MODEL ACCURACY 
y_pred = rf_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
r2 = r2_score(y_test, rf_pred)

print("MODEL ACCURACY REPORT")
print("="*50)
print(f"R² Score:        {r2:.3f} ")
print(f"RMSE:            {rmse:.3f} ")
print(f"Test samples:    {len(X_test)}")
print(f"Feature importance: {dict(zip(X.columns, rf_model.feature_importances_.round(3)))}")

MODEL ACCURACY REPORT
R² Score:        0.790 
RMSE:            0.075 
Test samples:    4000
Feature importance: {'hour': 0.234, 'is_weekend': 0.032, 'traffic_density_enc': 0.124, 'weather_enc': 0.246, 'cause_enc': 0.364}


In [17]:
# USER INPUT PREDICTION FUNCTION
def predict_risk(hour, is_weekend, traffic_density, weather, cause):
    """Predict risk_score from user input"""
    traffic_enc = label_encoders['traffic_density'].transform([traffic_density.lower()])[0]
    weather_enc = label_encoders['weather'].transform([weather.lower()])[0]
    cause_enc = label_encoders['cause'].transform([cause.lower()])[0]
    
    # Predict
    sample = pd.DataFrame({
        'hour': [hour], 'is_weekend': [is_weekend],
        'traffic_density_enc': [traffic_enc],
        'weather_enc': [weather_enc],
        'cause_enc': [cause_enc]
    })
    
    risk = rf_model.predict(sample)[0]
    
    # Risk level
    if risk > 0.7: level = "🚨 HIGH RISK"
    elif risk > 0.4: level = "⚠️ MEDIUM RISK" 
    else: level = "✅ LOW RISK"
    
    return risk, level


In [23]:
#  INTERACTIVE USER INPUT LOOP
print("\n" + "="*50)
print(" INTERACTIVE RISK PREDICTOR ")
print("Model Accuracy: R² =", round(r2, 3))
print("="*50)

while True:
    print("\n Enter accident scenario:")
    try:
        hour = int(input("Hour (0-23): "))
        weekend = int(input("Weekend? (0=No, 1=Yes): "))
        traffic = input("Traffic (low/medium/high): ").strip().lower()
        weather = input("Weather (clear/rain/fog): ").strip().lower()
        cause = input("Cause (overspeeding/drunk driving/distraction/poor road/weather): ").strip().lower()
        
        risk_score, risk_level = predict_risk(hour, weekend, traffic, weather, cause)
        print(f"\n PREDICTED: {risk_level} (Score: {risk_score:.3f})")
        
    except ValueError as e:
        print(f"Invalid input: {e}")
    
    if input("\nAnother prediction? (y/n): ").lower() != 'y':
        print("Model ready for production!")
        break


 INTERACTIVE RISK PREDICTOR 
Model Accuracy: R² = 0.79

 Enter accident scenario:


Hour (0-23):  3
Weekend? (0=No, 1=Yes):  0
Traffic (low/medium/high):  low
Weather (clear/rain/fog):  clear
Cause (overspeeding/drunk driving/distraction/poor road/weather):  drunk driving



 PREDICTED: ⚠️ MEDIUM RISK (Score: 0.631)



Another prediction? (y/n):  y



 Enter accident scenario:


Hour (0-23):  2
Weekend? (0=No, 1=Yes):  1
Traffic (low/medium/high):  low
Weather (clear/rain/fog):  rain
Cause (overspeeding/drunk driving/distraction/poor road/weather):  overspeeding



 PREDICTED: 🚨 HIGH RISK (Score: 0.797)



Another prediction? (y/n):  n


Model ready for production!


In [25]:
from scipy.stats import pearsonr

numeric_cols = ['hour', 'is_weekend', 'lanes']  
print("CORRELATION TESTS")
print("=" * 50)
for col in numeric_cols:
    r, p = pearsonr(df[col], df['risk_score'])
    print(f"{col} vs risk_score:  correlation = {r:.3f},  p-value = {p:.4f}")

CORRELATION TESTS
hour vs risk_score:  correlation = 0.014,  p-value = 0.0456
is_weekend vs risk_score:  correlation = 0.147,  p-value = 0.0000
lanes vs risk_score:  correlation = 0.004,  p-value = 0.5779


In [27]:
from scipy.stats import f_oneway

categorical_cols = ['traffic_density', 'weather', 'cause']  
print("ANOVA TESTS ")
print("=" * 50)
for col in categorical_cols:
    groups = [group['risk_score'].values for name, group in df.groupby(col)]
    f_stat, p = f_oneway(*groups)
    print(f"{col} vs risk_score:  F-statistic = {f_stat:.3f},  p-value = {p:.4f}")

ANOVA TESTS 
traffic_density vs risk_score:  F-statistic = 1028.425,  p-value = 0.0000
weather vs risk_score:  F-statistic = 2505.980,  p-value = 0.0000
cause vs risk_score:  F-statistic = 2072.904,  p-value = 0.0000


In [29]:
from scipy.stats import chi2_contingency

category_pairs = [
    ('cause', 'accident_severity'),
    ('traffic_density', 'road_type'),
    ('weather', 'cause')
]

print("CHI-SQUARE TESTS")
print("=" * 50)
for col1, col2 in category_pairs:
    contingency_table = pd.crosstab(df[col1], df[col2])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    print(f"{col1} vs {col2}:  chi2 = {chi2:.3f},  p-value = {p:.4f},  dof = {dof}")

CHI-SQUARE TESTS
cause vs accident_severity:  chi2 = 1372.028,  p-value = 0.0000,  dof = 8
traffic_density vs road_type:  chi2 = 2.699,  p-value = 0.6093,  dof = 4
weather vs cause:  chi2 = 5.561,  p-value = 0.6963,  dof = 8
